# 10c -- MTMC Stages 4-5: Association + Evaluation

**Prerequisite**: attach **10b's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-10b-stage-3-faiss-indexing" -> add`

**This is the iteration loop** -- edit the tuning params cell and re-run in ~6 min.
No GPU needed, no data download, no model inference.

| Stage | What | Time |
|---|---|---|
| 4 | Cross-camera association (AQE + Louvain graph clustering) | ~5 min |
| 5 | Evaluation: IDF1, MOTA, HOTA | ~1 min |

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _major, _minor = _cap.strip().split(".")
        _sm = int(_major) * 10 + int(_minor)
        if _sm < 70:
            print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) — installing compatible PyTorch ...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu124",
            ])
            print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/people-tracking", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

try:
    import trackeval; print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 Dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("trackeval", "trackeval"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Load Checkpoint from 10b

Finds `checkpoint.tar.gz` in `/kaggle/input/mtmc-10b-stage-3-faiss-indexing/` and extracts it.

In [ ]:
PREV_SLUG = "mtmc-10b-stage-3-faiss-indexing"
PREV_INPUT = Path("/kaggle/input") / PREV_SLUG

if not PREV_INPUT.exists():
    for p in Path("/kaggle/input").iterdir():
        if PREV_SLUG in p.name or "stage3" in p.name or "10b" in p.name:
            PREV_INPUT = p; break

cp = PREV_INPUT / "checkpoint.tar.gz"

# Fallback: download via Kaggle API if not found in mounted input
if not cp.exists():
    print(f"checkpoint.tar.gz not found at {cp} -- attempting API download")
    import subprocess as _sp
    _dl_dir = Path("/tmp/kaggle_dl")
    _dl_dir.mkdir(parents=True, exist_ok=True)
    _r = _sp.run(
        ["kaggle", "kernels", "output",
         "gumfreddy/mtmc-10b-stage-3-faiss-indexing",
         "--file", "checkpoint.tar.gz",
         "-p", str(_dl_dir)],
        capture_output=True, text=True
    )
    print(_r.stdout); print(_r.stderr)
    cp = _dl_dir / "checkpoint.tar.gz"

assert cp.exists(), f"checkpoint.tar.gz not found at {cp}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {cp.stat().st_size/1024**2:.1f} MB ...")
with tarfile.open(str(cp), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR  = EXTRACT_DIR / RUN_NAME
SECONDARY_EMBEDDINGS_PATH = RUN_DIR / "stage2" / "embeddings_secondary.npy"
if not SECONDARY_EMBEDDINGS_PATH.exists():
    print(f"\u26a0 Secondary embeddings not found at {SECONDARY_EMBEDDINGS_PATH}")
    SECONDARY_EMBEDDINGS_PATH = None

TERTIARY_EMBEDDINGS_PATH = RUN_DIR / "stage2" / "embeddings_tertiary.npy"
if not TERTIARY_EMBEDDINGS_PATH.exists():
    print(f"\u26a0 Tertiary embeddings not found at {TERTIARY_EMBEDDINGS_PATH}")
    TERTIARY_EMBEDDINGS_PATH = None

def secondary_embedding_overrides(weight):
    if SECONDARY_EMBEDDINGS_PATH is None or weight <= 0.0:
        return []
    return [
        "--override", f"stage4.association.secondary_embeddings.path={SECONDARY_EMBEDDINGS_PATH}",
        "--override", f"stage4.association.secondary_embeddings.weight={weight}",
    ]

def tertiary_embedding_overrides(weight):
    if TERTIARY_EMBEDDINGS_PATH is None or weight <= 0.0:
        return []
    return [
        "--override", f"stage4.association.tertiary_embeddings.path={TERTIARY_EMBEDDINGS_PATH}",
        "--override", f"stage4.association.tertiary_embeddings.weight={weight}",
    ]

# GT annotations are in the repo under data/raw/cityflowv2/*/gt/gt.txt
# They were force-committed despite data/ being gitignored.
_repo_gt = PROJECT / "data" / "raw" / "cityflowv2"
if any((_repo_gt / cam / "gt" / "gt.txt").exists()
       for cam in ["S01_c001", "S01_c002", "S01_c003",
                   "S02_c006", "S02_c007", "S02_c008"]):
    GT_DIR = str(_repo_gt)
    print(f"\u2713 GT annotations at {GT_DIR}")
else:
    GT_DIR = ""
    print("WARNING: GT files not found in repo — eval will skip metrics")
print(f"\u2713 Checkpoint extracted -- run: {RUN_NAME}")
for s in ["stage1", "stage2", "stage3"]:
    d = RUN_DIR / s
    if d.exists(): print(f"  {s}: {len(list(d.rglob('*')))} files")
print(f"  GT dir: {GT_DIR}")
print(f"  secondary embeddings: {SECONDARY_EMBEDDINGS_PATH}")
print(f"  tertiary embeddings: {TERTIARY_EMBEDDINGS_PATH}")


## 3. Tuning Parameters

**Edit these values** then re-run the cells below (~6 min). No need to re-run 10a or 10b.

In [ ]:
# ---- Stage 4: Cross-camera association -------------------------------------
# v52 association baseline on restored baseline features
AQE_K              = 3
SIM_THRESH         = 0.50
SOLVER             = "cc"
ALGORITHM          = "conflict_free_cc"
LOUVAIN_RES        = 0.70  # fallback (unused with cfcc)

# Weights — v80 appearance-heavy vehicle baseline
APPEARANCE_WEIGHT  = 0.70
HSV_WEIGHT         = 0.0
ST_WEIGHT          = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)

# Bridge pruning stays disabled; gallery/orphan thresholds match the v52 baseline
BRIDGE_PRUNE       = 0.0
MAX_COMP_SIZE      = 12
FIC_REG            = 0.50
GALLERY_THRESH     = 0.48
ORPHAN_MATCH_THRESH = 0.38

# Intra-camera merge (kept as the notebook baseline)
INTRA_MERGE        = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP    = 30

# Optional score-level fusion and multi-query blending
FUSION_WEIGHT      = 0.30 if SECONDARY_EMBEDDINGS_PATH is not None else 0.00
FUSION_WEIGHT_TERTIARY = 0.15 if TERTIARY_EMBEDDINGS_PATH is not None else 0.00
MULTI_QUERY_WEIGHT = 0.00

# v54-v57: camera_bias + zone_model BOTH HURT -> disabled
CAMERA_BIAS        = False
CAMERA_BIAS_ITERS  = 2
ZONE_MODEL         = False
ZONE_BONUS         = 0.06
ZONE_PENALTY       = 0.04
# Hierarchical: disabled (v54-56,v62: -1.0 to -5.1pp)
HIERARCHICAL       = False
HIER_CENTROID_TH   = 0.45
HIER_MERGE_TH      = 0.45
HIER_ORPHAN_TH     = 0.40

# K-reciprocal reranking baseline: off
RERANKING          = False
RERANKING_K1       = 20
RERANKING_K2       = 6
RERANKING_LAMBDA   = 0.5

# Camera-pair similarity normalization
CAMERA_PAIR_NORM   = False

# AFLink control defaults for the single baseline run
AFLINK_ENABLED     = False
AFLINK_MAX_TIME_GAP_FRAMES = 150
AFLINK_MAX_SPATIAL_GAP_PX = 150.0
AFLINK_MIN_DIRECTION_COS = 0.85
AFLINK_MIN_VELOCITY_RATIO = 0.5
AFLINK_VELOCITY_WINDOW = 5

# ---- Stage 5: Evaluation ----------------------------------------------------
MTMC_ONLY = False

print("Stage 4 params (v52 association baseline on restored features):")
print(f"  aqe_k={AQE_K}  sim_thresh={SIM_THRESH}  solver={SOLVER}  graph_algorithm={ALGORITHM}  appearance_weight={APPEARANCE_WEIGHT}")
print(f"  hsv_weight={HSV_WEIGHT}  st_weight={ST_WEIGHT}  bridge_prune={BRIDGE_PRUNE}  max_comp_size={MAX_COMP_SIZE}")
print(f"  fic_reg={FIC_REG}  gallery_thresh={GALLERY_THRESH}  orphan_match_threshold={ORPHAN_MATCH_THRESH}")
print(f"  intra_merge={INTRA_MERGE}  intra_thresh={INTRA_MERGE_THRESH}  intra_gap={INTRA_MERGE_GAP}")
print(
    f"  fusion_weight_secondary={FUSION_WEIGHT}  fusion_weight_tertiary={FUSION_WEIGHT_TERTIARY}  "
    f"multi_query_weight={MULTI_QUERY_WEIGHT}"
)
print(f"  camera_bias={CAMERA_BIAS}  zone_model={ZONE_MODEL}  hierarchical={HIERARCHICAL}  camera_pair_norm={CAMERA_PAIR_NORM}")
print(f"Stage 5: mtmc_only_submission={MTMC_ONLY}, gt_frame_clip=True, gt_zone_filter=True")

## 4. Run Stages 4-5

In [ ]:
os.chdir(str(PROJECT))
cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "4,5",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    # Stage 4: Association (v52 baseline control; AFLink disabled by default)
    "--override", f"stage4.association.query_expansion.k={AQE_K}",
    "--override", "stage4.association.query_expansion.alpha=5.0",
    "--override", "stage4.association.query_expansion.dba=false",
    "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    "--override", f"stage4.association.solver={SOLVER}",
    "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
    "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
    "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
    "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
    "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
    "--override", f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
    "--override", "stage4.association.mutual_nn.top_k_per_query=20",
    "--override", "stage4.association.fic.enabled=true",
    "--override", f"stage4.association.fic.regularisation={FIC_REG}",
    "--override", f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
    "--override", f"stage4.association.reranking.k1={RERANKING_K1}",
    "--override", f"stage4.association.reranking.k2={RERANKING_K2}",
    "--override", f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
    "--override", f"stage4.association.camera_pair_norm.enabled={str(CAMERA_PAIR_NORM).lower()}",
    "--override", "stage4.association.fac.enabled=false",
    "--override", f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
    "--override", f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
    "--override", f"stage4.association.aflink.enabled={str(AFLINK_ENABLED).lower()}",
    "--override", f"stage4.association.aflink.max_time_gap_frames={AFLINK_MAX_TIME_GAP_FRAMES}",
    "--override", f"stage4.association.aflink.max_spatial_gap_px={AFLINK_MAX_SPATIAL_GAP_PX}",
    "--override", f"stage4.association.aflink.min_direction_cos={AFLINK_MIN_DIRECTION_COS}",
    "--override", f"stage4.association.aflink.min_velocity_ratio={AFLINK_MIN_VELOCITY_RATIO}",
    "--override", f"stage4.association.aflink.velocity_window={AFLINK_VELOCITY_WINDOW}",
    # Optional score-level fusion with secondary and tertiary embeddings
    *secondary_embedding_overrides(FUSION_WEIGHT),
    *tertiary_embedding_overrides(FUSION_WEIGHT_TERTIARY),
    # Camera bias + zone model: disabled
    "--override", f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
    "--override", f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
    "--override", f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
    "--override", "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
    "--override", f"stage4.association.zone_model.bonus={ZONE_BONUS}",
    "--override", f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
    # Hierarchical: disabled
    "--override", f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
    "--override", f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
    "--override", f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
    "--override", f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
    "--override", "stage4.association.hierarchical.max_merge_size=12",
    # Intra-camera merge
    "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
    "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
    "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
    "--override", "stage4.association.gallery_expansion.enabled=true",
    "--override", f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
    "--override", f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
    "--override", f"stage4.association.weights.length_weight_power=0.3",
    "--override", f"stage4.association.temporal_overlap.enabled=true",
    "--override", f"stage4.association.temporal_overlap.bonus=0.05",
    "--override", f"stage4.association.temporal_overlap.max_mean_time=5.0",
    # Stage 5: Post-processing
    "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
    "--override", "stage5.stationary_filter.enabled=true",
    "--override", "stage5.stationary_filter.min_displacement_px=150",
    "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
    "--override", "stage5.min_submission_confidence=0.15",
    "--override", "stage5.cross_id_nms_iou=0.40",
    "--override", "stage5.min_trajectory_confidence=0.30",
    "--override", "stage5.min_trajectory_frames=40",
    "--override", "stage5.track_edge_trim.enabled=false",
    "--override", "stage5.track_smoothing.enabled=false",
    "--override", "stage5.gt_frame_clip=true",
    "--override", "stage5.gt_zone_filter=true",
]
if GT_DIR:
    cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
else:
    print("WARNING: GT_DIR is empty — eval will skip metric computation")
print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"✗ FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"✓ Stages 4-5 done in {elapsed/60:.1f} min")

## 5. Results

In [ ]:
stage5_dir = RUN_DIR / "stage5"

def _pct(v):
    return f"{v:.1%}" if isinstance(v, float) else str(v)

metrics_files = sorted(stage5_dir.glob("metrics_*.json")) if stage5_dir.exists() else []
if metrics_files:
    print("=" * 65)
    print("EVALUATION RESULTS")
    print("=" * 65)
    for mf in metrics_files:
        m = json.loads(mf.read_text())
        m = m.get("metrics", m)
        cam = mf.stem.replace("metrics_", "")
        idf1 = _pct(m.get("IDF1", m.get("idf1", "?")))
        mota = _pct(m.get("MOTA", m.get("mota", "?")))
        hota = _pct(m.get("HOTA", m.get("hota", "?")))
        idsw = m.get("ID_Sw", m.get("id_switches", "?"))
        print(f"  {cam:12s}  IDF1={idf1}  MOTA={mota}  HOTA={hota}  IDsw={idsw}")

for fname in ["summary.json", "evaluation_report.json"]:
    sf = stage5_dir / fname
    if sf.exists():
        s = json.loads(sf.read_text())
        print("-" * 65 + "\n  GLOBAL:")
        for k in ["IDF1","MOTA","HOTA","ID_Sw","idf1","mota","hota","id_switches","mtmc_idf1"]:
            v = s.get(k)
            if v is not None: print(f"    {k}: {_pct(v)}")
        break

if not metrics_files:
    print("No metrics files found -- check stage5 output dir:", stage5_dir)

# Copy evaluation report to /kaggle/working/ for easy download
import shutil as _shutil
for _fname in ["evaluation_report.json", "summary.json"]:
    _src = stage5_dir / _fname
    if _src.exists():
        _shutil.copy2(str(_src), str(Path("/kaggle/working") / _fname))
        print(f"Copied {_fname} to /kaggle/working/")

## 6. AFLink Pure Addon Test

Runs the v52 association baseline on restored baseline features, then compares three AFLink-only addon variants against the no-AFLink control.

This keeps the v52 base association parameters fixed and isolates AFLink impact.

In [ ]:
# ============================================================
# AFLink pure addon test on the fixed v52 association baseline
# ============================================================
AFLINK_ADDON_ENABLED = True

if AFLINK_ADDON_ENABLED:
    import shutil as _shutil

    PRIMARY_METRIC = "mtmc_idf1"
    addon_configs = [
        {
            "label": "control_no_aflink",
            "run_name": RUN_NAME,
            "reuse_existing_report": True,
            "aflink_enabled": False,
            "aflink_max_spatial_gap": None,
            "aflink_min_direction_cos": None,
            "aflink_min_velocity_ratio": None,
        },
        {
            "label": "aflink_gap150_dir085",
            "run_name": f"{RUN_NAME}_aflink_gap150_dir085",
            "reuse_existing_report": False,
            "aflink_enabled": True,
            "aflink_max_spatial_gap": 150.0,
            "aflink_min_direction_cos": 0.85,
            "aflink_min_velocity_ratio": 0.5,
        },
        {
            "label": "aflink_gap100_dir090",
            "run_name": f"{RUN_NAME}_aflink_gap100_dir090",
            "reuse_existing_report": False,
            "aflink_enabled": True,
            "aflink_max_spatial_gap": 100.0,
            "aflink_min_direction_cos": 0.90,
            "aflink_min_velocity_ratio": 0.5,
        },
        {
            "label": "aflink_gap200_dir070",
            "run_name": f"{RUN_NAME}_aflink_gap200_dir070",
            "reuse_existing_report": False,
            "aflink_enabled": True,
            "aflink_max_spatial_gap": 200.0,
            "aflink_min_direction_cos": 0.70,
            "aflink_min_velocity_ratio": 0.5,
        },
    ]

    def _load_addon_metrics(report_path):
        metrics = {
            "idf1": 0.0,
            PRIMARY_METRIC: 0.0,
            "mota": 0.0,
            "hota": 0.0,
            "mtmc_mota": 0.0,
            "score": 0.0,
        }
        if not report_path.exists():
            return metrics

        payload = json.loads(report_path.read_text(encoding="utf-8"))
        raw_metrics = payload.get("metrics", payload)
        idf1 = raw_metrics.get("mtmc_idf1", raw_metrics.get("IDF1", raw_metrics.get("idf1", 0.0)))
        mota = raw_metrics.get("MOTA", raw_metrics.get("mota", 0.0))
        hota = raw_metrics.get("HOTA", raw_metrics.get("hota", 0.0))
        details = payload.get("details", {})
        mtmc_mota = details.get("mtmc_mota", mota)
        metrics.update({
            "idf1": raw_metrics.get("IDF1", raw_metrics.get("idf1", 0.0)),
            PRIMARY_METRIC: raw_metrics.get("mtmc_idf1", idf1),
            "mota": mota,
            "hota": hota,
            "mtmc_mota": mtmc_mota,
            "score": raw_metrics.get("mtmc_idf1", idf1),
        })
        return metrics

    def _ensure_upstream_links(run_name):
        run_dir = DATA_OUT / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        for stage_sub in ("stage1", "stage2", "stage3"):
            src = DATA_OUT / RUN_NAME / stage_sub
            dst = run_dir / stage_sub
            if src.exists() and not dst.exists():
                dst.symlink_to(src)
        return run_dir

    def _build_cmd(run_name, config):
        cmd = [
            sys.executable, "scripts/run_pipeline.py",
            "--config", "configs/default.yaml",
            "--dataset-config", "configs/datasets/cityflowv2.yaml",
            "--stages", "4,5",
            "--override", f"project.run_name={run_name}",
            "--override", f"project.output_dir={DATA_OUT}",
            "--override", f"stage4.association.query_expansion.k={AQE_K}",
            "--override", "stage4.association.query_expansion.alpha=5.0",
            "--override", "stage4.association.query_expansion.dba=false",
            "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
            "--override", f"stage4.association.solver={SOLVER}",
            "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
            "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
            "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
            "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
            "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
            "--override", "stage4.association.mutual_nn.top_k_per_query=20",
            "--override", "stage4.association.fic.enabled=true",
            "--override", f"stage4.association.fic.regularisation={FIC_REG}",
            "--override", f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
            "--override", f"stage4.association.reranking.k1={RERANKING_K1}",
            "--override", f"stage4.association.reranking.k2={RERANKING_K2}",
            "--override", f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
            "--override", f"stage4.association.camera_pair_norm.enabled={str(CAMERA_PAIR_NORM).lower()}",
            "--override", "stage4.association.fac.enabled=false",
            "--override", f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
            "--override", f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
            *secondary_embedding_overrides(FUSION_WEIGHT),
            *tertiary_embedding_overrides(FUSION_WEIGHT_TERTIARY),
            "--override", f"stage4.association.aflink.enabled={str(config['aflink_enabled']).lower()}",
            "--override", "stage4.association.aflink.max_time_gap_frames=150",
            "--override", f"stage4.association.aflink.max_spatial_gap_px={config['aflink_max_spatial_gap'] if config['aflink_max_spatial_gap'] is not None else 150.0}",
            "--override", f"stage4.association.aflink.min_direction_cos={config['aflink_min_direction_cos'] if config['aflink_min_direction_cos'] is not None else 0.85}",
            "--override", f"stage4.association.aflink.min_velocity_ratio={config['aflink_min_velocity_ratio'] if config['aflink_min_velocity_ratio'] is not None else 0.5}",
            "--override", "stage4.association.aflink.velocity_window=5",
            "--override", "stage4.association.gallery_expansion.enabled=true",
            "--override", f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
            "--override", f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
            "--override", f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
            "--override", f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
            "--override", f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
            "--override", "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
            "--override", f"stage4.association.zone_model.bonus={ZONE_BONUS}",
            "--override", f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
            "--override", f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
            "--override", f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
            "--override", f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
            "--override", f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
            "--override", "stage4.association.hierarchical.max_merge_size=12",
            "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
            "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
            "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
            "--override", "stage4.association.weights.length_weight_power=0.3",
            "--override", "stage4.association.temporal_overlap.enabled=true",
            "--override", "stage4.association.temporal_overlap.bonus=0.05",
            "--override", "stage4.association.temporal_overlap.max_mean_time=5.0",
            "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
            "--override", "stage5.stationary_filter.enabled=true",
            "--override", "stage5.stationary_filter.min_displacement_px=150",
            "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
            "--override", "stage5.min_submission_confidence=0.15",
            "--override", "stage5.cross_id_nms_iou=0.40",
            "--override", "stage5.min_trajectory_confidence=0.30",
            "--override", "stage5.min_trajectory_frames=40",
            "--override", "stage5.track_edge_trim.enabled=false",
            "--override", "stage5.track_smoothing.enabled=false",
            "--override", "stage5.gt_frame_clip=true",
            "--override", "stage5.gt_zone_filter=true",
        ]
        if GT_DIR:
            cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
        return cmd

    print("Running AFLink pure addon test on the fixed v52 association baseline.")
    print(
        f"Base config: sim={SIM_THRESH:.2f}, app={APPEARANCE_WEIGHT:.2f}, fic={FIC_REG:.2f}, "
        f"aqe={AQE_K}, gallery={GALLERY_THRESH:.2f}, orphan={ORPHAN_MATCH_THRESH:.2f}"
    )
    print("=" * 96)

    results = []
    for config in addon_configs:
        run_name = config["run_name"]
        report = DATA_OUT / run_name / "stage5" / "evaluation_report.json"
        if config["reuse_existing_report"] and report.exists():
            metrics = _load_addon_metrics(report)
            status = "REUSE"
            elapsed = 0.0
            returncode = 0
        else:
            _ensure_upstream_links(run_name)
            cmd = _build_cmd(run_name, config)
            t0 = time.time()
            run_result = subprocess.run(cmd, cwd=str(PROJECT), capture_output=True, text=True)
            elapsed = time.time() - t0
            returncode = run_result.returncode
            status = "OK" if returncode == 0 else "FAIL"
            metrics = _load_addon_metrics(report)
            if returncode != 0:
                print("  stdout tail:")
                for line in run_result.stdout.splitlines()[-10:]:
                    print(f"    {line}")
                print("  stderr tail:")
                for line in run_result.stderr.splitlines()[-10:]:
                    print(f"    {line}")

        result = {
            "label": config["label"],
            "run_name": run_name,
            "status": status,
            "returncode": returncode,
            "time": elapsed,
            "aflink_enabled": config["aflink_enabled"],
            "aflink_max_spatial_gap": config["aflink_max_spatial_gap"],
            "aflink_min_direction_cos": config["aflink_min_direction_cos"],
            "aflink_min_velocity_ratio": config["aflink_min_velocity_ratio"],
            **metrics,
        }
        results.append(result)
        print(
            f"  [{status}] {config['label']:<22} mtmc_idf1={result[PRIMARY_METRIC]:.4f} "
            f"IDF1={result['idf1']:.4f} HOTA={result['hota']:.4f} ({elapsed/60:.1f}m)"
        )

    results.sort(key=lambda item: item[PRIMARY_METRIC], reverse=True)
    print("\n" + "=" * 96)
    print("AFLINK PURE ADDON RESULTS")
    print("=" * 96)
    print(f"{'label':<22} {'status':<7} {'mtmc_idf1':>10} {'IDF1':>8} {'HOTA':>8} {'gap':>8} {'dir':>8}")
    for row in results:
        gap = "-" if row['aflink_max_spatial_gap'] is None else f"{row['aflink_max_spatial_gap']:.1f}"
        direction = "-" if row['aflink_min_direction_cos'] is None else f"{row['aflink_min_direction_cos']:.2f}"
        print(
            f"{row['label']:<22} {row['status']:<7} {row[PRIMARY_METRIC]:>10.4f} {row['idf1']:>8.4f} "
            f"{row['hota']:>8.4f} {gap:>8} {direction:>8}"
        )

    best = results[0]
    print(
        f"BEST: {best['label']} -> mtmc_idf1={best[PRIMARY_METRIC]:.4f} "
        f"IDF1={best['idf1']:.4f} HOTA={best['hota']:.4f}"
    )

    payload = {
        "experiment": "aflink_pure_addon_v52_baseline",
        "base_params": {
            "sim_thresh": SIM_THRESH,
            "appearance_weight": APPEARANCE_WEIGHT,
            "fic_reg": FIC_REG,
            "aqe_k": AQE_K,
            "gallery_thresh": GALLERY_THRESH,
            "orphan_match_threshold": ORPHAN_MATCH_THRESH,
        },
        "results": results,
        "best": best,
    }
    results_path = DATA_OUT / "aflink_addon_results.json"
    with open(results_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
    print(f"Saved AFLink addon results to {results_path}")

    kaggle_out = Path("/kaggle/working")
    if kaggle_out.exists():
        _shutil.copy2(str(results_path), str(kaggle_out / "aflink_addon_results.json"))
        print(f"Copied aflink_addon_results.json to {kaggle_out}")
else:
    print("AFLink addon test disabled. Set AFLINK_ADDON_ENABLED = True to run.")

## 7. Association Parameter Sweep

Sequentially tunes association parameters from the restored v80 baseline
(camera_bn off + color_augment off + camera_tta off + power_norm alpha 0.0 +
concat_patch on + samples_per_tracklet=48), then runs intra-camera merge
structural experiments on the tuned best configuration.

In [ ]:
# --- Association parameter sweep from the restored baseline (disabled for AFLink addon test) ---
FEATURE_TEST_ENABLED = False

if FEATURE_TEST_ENABLED:
    import shutil as _shutil

    PRIMARY_METRIC = "mtmc_idf1"
    BASE_FIC_REG = 0.50
    BASE_GALLERY_THRESH = 0.48
    BASE_ORPHAN_THRESH = 0.38

    current_params = {
        "sim_thresh": SIM_THRESH,
        "appearance_w": APPEARANCE_WEIGHT,
        "fic_reg": BASE_FIC_REG,
        "fusion_weight": FUSION_WEIGHT,
        "fusion_weight_tertiary": FUSION_WEIGHT_TERTIARY,
        "multi_query_weight": MULTI_QUERY_WEIGHT,
        "aqe_k": AQE_K,
        "gallery_thresh": BASE_GALLERY_THRESH,
        "orphan_thresh": BASE_ORPHAN_THRESH,
        "intra_merge": INTRA_MERGE,
        "intra_merge_thresh": INTRA_MERGE_THRESH,
        "intra_merge_gap": INTRA_MERGE_GAP,
        "reranking": RERANKING,
        "reranking_k1": RERANKING_K1,
        "reranking_k2": RERANKING_K2,
        "reranking_lambda": RERANKING_LAMBDA,
        "camera_pair_norm": CAMERA_PAIR_NORM,
        "camera_pair_norm_min_pairs": 10,
        "aflink_max_spatial_gap": 200.0,
        "aflink_min_direction_cos": 0.7,
    }

    sweep_plan = [
        {
            "name": "Sweep 1 - sim_thresh",
            "key": "sim_thresh",
            "options": [0.53, 0.52, 0.54, 0.50, 0.55, 0.48, 0.58, 0.60, 0.45],
            "candidate_label": lambda value: f"sim_thresh={value:.2f}",
            "apply": lambda params, value: params.update({"sim_thresh": value}),
        },
        {
            "name": "Sweep 2 - appearance_weight",
            "key": "appearance_w",
            "options": [0.70, 0.65, 0.75, 0.60, 0.80, 0.85],
            "candidate_label": lambda value: f"appearance_weight={value:.2f}",
            "apply": lambda params, value: params.update({"appearance_w": value}),
        },
        {
            "name": "Sweep 3 - FIC regularisation",
            "key": "fic_reg",
            "options": [0.10, 0.05, 0.20, 0.01, 0.50, 1.0],
            "candidate_label": lambda value: f"fic_reg={value:.2f}",
            "apply": lambda params, value: params.update({"fic_reg": value}),
        },
        {
            "name": "Sweep 4 - AQE k",
            "key": "aqe_k",
            "options": [3, 2, 4, 1, 5],
            "candidate_label": lambda value: f"aqe_k={value}",
            "apply": lambda params, value: params.update({"aqe_k": value}),
        },
        {
            "name": "Sweep 5 - gallery threshold + orphan match",
            "key": "gallery_orphan",
            "options": [(0.50, 0.40), (0.48, 0.38), (0.52, 0.42), (0.45, 0.35), (0.55, 0.45)],
            "candidate_label": lambda value: (
                f"gallery_thresh={value[0]:.2f}, orphan_match={value[1]:.2f}"
            ),
            "apply": lambda params, value: params.update({
                "gallery_thresh": value[0],
                "orphan_thresh": value[1],
            }),
        },
        {
            "name": "Sweep 6 - fusion_weight (secondary embeddings)",
            "key": "fusion_weight",
            "options": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
            "candidate_label": lambda value: f"fusion_weight={value:.1f}",
            "apply": lambda params, value: params.update({"fusion_weight": value}),
        },
        {
            "name": "Sweep 7 - multi_query.weight",
            "key": "multi_query_weight",
            "options": [0.0, 0.3, 0.5, 0.7, 1.0],
            "candidate_label": lambda value: f"multi_query.weight={value:.1f}",
            "apply": lambda params, value: params.update({"multi_query_weight": value}),
        },
        {
            "name": "Sweep 8 - AFLink max_spatial_gap_px",
            "key": "aflink_max_spatial_gap",
            "options": [150.0, 200.0, 300.0],
            "candidate_label": lambda value: f"aflink.max_spatial_gap_px={value:.1f}",
            "apply": lambda params, value: params.update({"aflink_max_spatial_gap": value}),
        },
        {
            "name": "Sweep 9 - AFLink min_direction_cos",
            "key": "aflink_min_direction_cos",
            "options": [0.5, 0.7, 0.85],
            "candidate_label": lambda value: f"aflink.min_direction_cos={value:.2f}",
            "apply": lambda params, value: params.update({"aflink_min_direction_cos": value}),
        },
        {
            "name": "Sweep 10 - k-reciprocal reranking",
            "key": "reranking",
            "options": [
                {"label": "rerank_off", "overrides": {
                    "stage4.association.reranking.enabled": "false",
                    "stage4.association.reranking.k1": "20",
                    "stage4.association.reranking.k2": "6",
                    "stage4.association.reranking.lambda_value": "0.5",
                }},
                {"label": "rerank_l03", "overrides": {
                    "stage4.association.reranking.enabled": "true",
                    "stage4.association.reranking.k1": "20",
                    "stage4.association.reranking.k2": "6",
                    "stage4.association.reranking.lambda_value": "0.3",
                }},
                {"label": "rerank_l05", "overrides": {
                    "stage4.association.reranking.enabled": "true",
                    "stage4.association.reranking.k1": "20",
                    "stage4.association.reranking.k2": "6",
                    "stage4.association.reranking.lambda_value": "0.5",
                }},
                {"label": "rerank_l07", "overrides": {
                    "stage4.association.reranking.enabled": "true",
                    "stage4.association.reranking.k1": "20",
                    "stage4.association.reranking.k2": "6",
                    "stage4.association.reranking.lambda_value": "0.7",
                }},
                {"label": "rerank_l09", "overrides": {
                    "stage4.association.reranking.enabled": "true",
                    "stage4.association.reranking.k1": "20",
                    "stage4.association.reranking.k2": "6",
                    "stage4.association.reranking.lambda_value": "0.9",
                }},
                {"label": "rerank_k30_l05", "overrides": {
                    "stage4.association.reranking.enabled": "true",
                    "stage4.association.reranking.k1": "30",
                    "stage4.association.reranking.k2": "10",
                    "stage4.association.reranking.lambda_value": "0.5",
                }},
            ],
            "candidate_label": lambda value: value["label"],
            "apply": lambda params, value: params.update({
                "reranking": value["overrides"]["stage4.association.reranking.enabled"] == "true",
                "reranking_k1": int(value["overrides"]["stage4.association.reranking.k1"]),
                "reranking_k2": int(value["overrides"]["stage4.association.reranking.k2"]),
                "reranking_lambda": float(value["overrides"]["stage4.association.reranking.lambda_value"]),
            }),
        },
        {
            "name": "Sweep 11 - camera-pair similarity normalization",
            "key": "camera_pair_norm",
            "options": [
                {"label": "campair_norm_off", "overrides": {
                    "stage4.association.camera_pair_norm.enabled": "false",
                    "stage4.association.camera_pair_norm.min_pairs": "10",
                }},
                {"label": "campair_norm_on", "overrides": {
                    "stage4.association.camera_pair_norm.enabled": "true",
                    "stage4.association.camera_pair_norm.min_pairs": "10",
                }},
                {"label": "campair_norm_mp5", "overrides": {
                    "stage4.association.camera_pair_norm.enabled": "true",
                    "stage4.association.camera_pair_norm.min_pairs": "5",
                }},
                {"label": "campair_norm_mp20", "overrides": {
                    "stage4.association.camera_pair_norm.enabled": "true",
                    "stage4.association.camera_pair_norm.min_pairs": "20",
                }},
            ],
            "candidate_label": lambda value: value["label"],
            "apply": lambda params, value: params.update({
                "camera_pair_norm": value["overrides"]["stage4.association.camera_pair_norm.enabled"] == "true",
                "camera_pair_norm_min_pairs": int(value["overrides"]["stage4.association.camera_pair_norm.min_pairs"]),
            }),
        },
    ]

    structural_experiments = [
        {
            "name": "intra_merge_t070_gap120",
            "intra_merge": True,
            "intra_merge_thresh": 0.70,
            "intra_merge_gap": 120,
        },
        {
            "name": "intra_merge_t080_gap060",
            "intra_merge": True,
            "intra_merge_thresh": 0.80,
            "intra_merge_gap": 60,
        },
    ]

    def _tag_float(prefix, value):
        return f"{prefix}{value:.2f}".replace(".", "")

    def _build_run_name(prefix, params):
        merge_tag = "mergeon" if params["intra_merge"] else "mergeoff"
        parts = [
            RUN_NAME,
            prefix,
            _tag_float("sim", params["sim_thresh"]),
            _tag_float("app", params["appearance_w"]),
            _tag_float("fic", params["fic_reg"]),
            f"fus{params['fusion_weight']:.1f}",
            f"ter{params['fusion_weight_tertiary']:.1f}",
            f"mq{params['multi_query_weight']:.1f}",
            f"aqe{params['aqe_k']}",
            _tag_float("gal", params["gallery_thresh"]),
            _tag_float("orp", params["orphan_thresh"]),
            merge_tag,
            _tag_float("imt", params["intra_merge_thresh"]),
            f"img{params['intra_merge_gap']}",
        ]
        return "_".join(parts)

    def _ensure_upstream_links(run_name):
        run_dir = DATA_OUT / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        for stage_sub in ("stage1", "stage2", "stage3"):
            src = RUN_DIR / stage_sub
            dst = run_dir / stage_sub
            if src.exists() and not dst.exists():
                dst.symlink_to(src)
        return run_dir

    def _load_metrics(report_path):
        metrics = {
            "IDF1": 0.0,
            PRIMARY_METRIC: 0.0,
            "MOTA": 0.0,
            "HOTA": 0.0,
            "MTMC_MOTA": 0.0,
            "score": 0.0,
            "score_metric": PRIMARY_METRIC,
        }
        if not report_path.exists():
            return metrics

        payload = json.loads(report_path.read_text(encoding="utf-8"))
        raw_metrics = payload.get("metrics", payload)
        idf1 = raw_metrics.get("IDF1", raw_metrics.get("idf1", 0.0))
        mtmc_idf1 = raw_metrics.get(PRIMARY_METRIC, idf1)
        mota = raw_metrics.get("MOTA", raw_metrics.get("mota", 0.0))
        hota = raw_metrics.get("HOTA", raw_metrics.get("hota", 0.0))
        details = payload.get("details", {})
        mtmc_mota = details.get("mtmc_mota", mota)
        score_metric = PRIMARY_METRIC if mtmc_idf1 not in (None, 0.0) else "IDF1"
        score = mtmc_idf1 if score_metric == PRIMARY_METRIC else idf1
        metrics.update({
            "IDF1": idf1,
            PRIMARY_METRIC: mtmc_idf1,
            "MOTA": mota,
            "HOTA": hota,
            "MTMC_MOTA": mtmc_mota,
            "score": score,
            "score_metric": score_metric,
        })
        return metrics

    def _rank_key(result):
        return (result["score"], result["HOTA"], result["MOTA"], result["IDF1"])

    def _print_result(result):
        print(
            f"  [{result['status']}] {result['candidate']:<38} "
            f"{result['score_metric']}={result['score']:.3f} "
            f"IDF1={result['IDF1']:.3f} MOTA={result['MOTA']:.3f} "
            f"HOTA={result['HOTA']:.3f} ({result['time'] / 60:.1f} min)"
        )

    def _print_table(title, rows):
        print("\n" + "=" * 96)
        print(title)
        print("=" * 96)
        print(f"  {'candidate':<38} {'score':>7} {'metric':<10} {'IDF1':>7} {'mtmc':>7} {'MOTA':>7} {'HOTA':>7}")
        for row in sorted(rows, key=_rank_key, reverse=True):
            print(
                f"  {row['candidate']:<38} {row['score']:>7.3f} {row['score_metric']:<10} "
                f"{row['IDF1']:>7.3f} {row[PRIMARY_METRIC]:>7.3f} {row['MOTA']:>7.3f} {row['HOTA']:>7.3f}"
            )

    def _run_experiment(prefix, candidate, params):
        run_name = _build_run_name(prefix, params)
        _ensure_upstream_links(run_name)
        st_w = round(1.0 - params["appearance_w"] - HSV_WEIGHT, 4)
        cmd = [
            sys.executable, "scripts/run_pipeline.py",
            "--config", "configs/default.yaml",
            "--dataset-config", "configs/datasets/cityflowv2.yaml",
            "--stages", "4,5",
            "--override", f"project.run_name={run_name}",
            "--override", f"project.output_dir={DATA_OUT}",
            "--override", f"stage4.association.query_expansion.k={params['aqe_k']}",
            "--override", "stage4.association.query_expansion.dba=false",
            "--override", f"stage4.association.graph.similarity_threshold={params['sim_thresh']}",
            "--override", f"stage4.association.solver={SOLVER}",
            "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
            "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
            "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
            "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
            "--override", f"stage4.association.weights.vehicle.appearance={params['appearance_w']}",
            "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
            "--override", f"stage4.association.weights.vehicle.spatiotemporal={st_w}",
            "--override", "stage4.association.weights.length_weight_power=0.3",
            "--override", "stage4.association.mutual_nn.top_k_per_query=20",
            "--override", "stage4.association.fic.enabled=true",
            "--override", f"stage4.association.fic.regularisation={params['fic_reg']}",
            "--override", f"stage4.association.reranking.enabled={str(params['reranking']).lower()}",
            "--override", f"stage4.association.reranking.k1={params['reranking_k1']}",
            "--override", f"stage4.association.reranking.k2={params['reranking_k2']}",
            "--override", f"stage4.association.reranking.lambda_value={params['reranking_lambda']}",
            "--override", f"stage4.association.camera_pair_norm.enabled={str(params['camera_pair_norm']).lower()}",
            "--override", f"stage4.association.camera_pair_norm.min_pairs={params['camera_pair_norm_min_pairs']}",
            "--override", "stage4.association.fac.enabled=false",
            "--override", "stage4.association.fac.knn=20",
            "--override", "stage4.association.fac.learning_rate=0.5",
            "--override", "stage4.association.fac.beta=0.08",
            "--override", "stage4.association.gallery_expansion.enabled=true",
            "--override", f"stage4.association.gallery_expansion.threshold={params['gallery_thresh']}",
            "--override", f"stage4.association.gallery_expansion.orphan_match_threshold={params['orphan_thresh']}",
            *secondary_embedding_overrides(params['fusion_weight']),
            *tertiary_embedding_overrides(params['fusion_weight_tertiary']),
            "--override", f"stage4.association.multi_query.enabled={str(params['multi_query_weight'] > 0.0).lower()}",
            "--override", f"stage4.association.multi_query.weight={params['multi_query_weight']}",
            "--override", "stage4.association.aflink.enabled=true",
            "--override", "stage4.association.aflink.max_time_gap_frames=150",
            "--override", f"stage4.association.aflink.max_spatial_gap_px={params['aflink_max_spatial_gap']}",
            "--override", f"stage4.association.aflink.min_direction_cos={params['aflink_min_direction_cos']}",
            "--override", "stage4.association.aflink.min_velocity_ratio=0.5",
            "--override", "stage4.association.aflink.velocity_window=5",
            "--override", f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
            "--override", f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
            "--override", f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
            "--override", "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
            "--override", f"stage4.association.zone_model.bonus={ZONE_BONUS}",
            "--override", f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
            "--override", f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
            "--override", f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
            "--override", f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
            "--override", f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
            "--override", "stage4.association.hierarchical.max_merge_size=12",
            "--override", f"stage4.association.intra_camera_merge.enabled={str(params['intra_merge']).lower()}",
            "--override", f"stage4.association.intra_camera_merge.threshold={params['intra_merge_thresh']}",
            "--override", f"stage4.association.intra_camera_merge.max_time_gap={params['intra_merge_gap']}",
            "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
            "--override", "stage5.stationary_filter.enabled=true",
            "--override", "stage5.stationary_filter.min_displacement_px=150",
            "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
            "--override", "stage5.min_submission_confidence=0.15",
            "--override", "stage5.cross_id_nms_iou=0.40",
            "--override", "stage5.min_trajectory_confidence=0.30",
            "--override", "stage5.min_trajectory_frames=40",
            "--override", "stage5.track_edge_trim.enabled=false",
            "--override", "stage5.track_smoothing.enabled=false",
            "--override", "stage5.gt_frame_clip=true",
            "--override", "stage5.gt_zone_filter=true",
        ]
        if GT_DIR:
            cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]

        t0 = time.time()
        run_result = subprocess.run(cmd, cwd=str(PROJECT), capture_output=True, text=True)
        elapsed = time.time() - t0
        report = DATA_OUT / run_name / "stage5" / "evaluation_report.json"
        metrics = _load_metrics(report)
        result = {
            "run_name": run_name,
            "candidate": candidate,
            "status": "OK" if run_result.returncode == 0 else "FAIL",
            "returncode": run_result.returncode,
            "time": elapsed,
            **params,
            **metrics,
        }
        if run_result.returncode != 0:
            result["stdout_tail"] = run_result.stdout.splitlines()[-10:]
            result["stderr_tail"] = run_result.stderr.splitlines()[-10:]
            print("  stdout tail:")
            for line in result["stdout_tail"]:
                print(f"    {line}")
            print("  stderr tail:")
            for line in result["stderr_tail"]:
                print(f"    {line}")
        _print_result(result)
        return result

    print("Running sequential association sweep from the restored v80 baseline.")
    print(
        f"Base config: sim={current_params['sim_thresh']:.2f}, app={current_params['appearance_w']:.2f}, "
        f"fic={current_params['fic_reg']:.2f}, aqe={current_params['aqe_k']}, "
        f"gallery={current_params['gallery_thresh']:.2f}, orphan={current_params['orphan_thresh']:.2f}, "
        f"aflink_gap={current_params['aflink_max_spatial_gap']:.1f}, "
        f"aflink_dir={current_params['aflink_min_direction_cos']:.2f}, "
        f"fusion_secondary={current_params['fusion_weight']:.2f}, "
        f"fusion_tertiary={current_params['fusion_weight_tertiary']:.2f}, mq={current_params['multi_query_weight']:.2f}"
    )

    stage_winners = []
    all_results = []
    current_best_result = None

    for stage_index, stage in enumerate(sweep_plan, start=1):
        print("\n" + "=" * 96)
        print(stage["name"])
        print("=" * 96)
        stage_results = []
        for option_index, option in enumerate(stage["options"], start=1):
            trial_params = dict(current_params)
            stage["apply"](trial_params, option)
            candidate = stage["candidate_label"](option)
            prefix = f"assoc_s{stage_index}_c{option_index}"
            result = _run_experiment(prefix, candidate, trial_params)
            stage_results.append(result)
            all_results.append(result)

        _print_table(stage["name"] + " results", stage_results)
        best_stage_result = max(stage_results, key=_rank_key)
        current_best_result = best_stage_result
        current_params = {
            key: best_stage_result[key]
            for key in current_params
        }
        stage_winners.append({
            "stage": stage["name"],
            "best_candidate": best_stage_result["candidate"],
            "best_run_name": best_stage_result["run_name"],
            "score_metric": best_stage_result["score_metric"],
            "score": best_stage_result["score"],
            "IDF1": best_stage_result["IDF1"],
            PRIMARY_METRIC: best_stage_result[PRIMARY_METRIC],
            "MOTA": best_stage_result["MOTA"],
            "HOTA": best_stage_result["HOTA"],
            "params": dict(current_params),
        })
        print(
            f"Best after {stage['name']}: {best_stage_result['candidate']} -> "
            f"{best_stage_result['score_metric']}={best_stage_result['score']:.3f}, "
            f"IDF1={best_stage_result['IDF1']:.3f}, MOTA={best_stage_result['MOTA']:.3f}, "
            f"HOTA={best_stage_result['HOTA']:.3f}"
        )

    tuned_best_params = dict(current_params)
    tuned_best_result = current_best_result

    print("\n" + "=" * 96)
    print("Tuned parameter winner before structural experiments")
    print("=" * 96)
    print(
        f"sim_thresh={tuned_best_params['sim_thresh']:.2f}, "
        f"appearance_weight={tuned_best_params['appearance_w']:.2f}, "
        f"spatiotemporal_weight={round(1.0 - tuned_best_params['appearance_w'] - HSV_WEIGHT, 4):.4f}, "
        f"fic_reg={tuned_best_params['fic_reg']:.2f}, aqe_k={tuned_best_params['aqe_k']}, "
        f"gallery_thresh={tuned_best_params['gallery_thresh']:.2f}, "
        f"orphan_match={tuned_best_params['orphan_thresh']:.2f}, "
        f"aflink_gap={tuned_best_params['aflink_max_spatial_gap']:.1f}, "
        f"aflink_dir={tuned_best_params['aflink_min_direction_cos']:.2f}"
    )
    print(
        f"Best tuned metrics: {tuned_best_result['score_metric']}={tuned_best_result['score']:.3f}, "
        f"IDF1={tuned_best_result['IDF1']:.3f}, MOTA={tuned_best_result['MOTA']:.3f}, "
        f"HOTA={tuned_best_result['HOTA']:.3f}"
    )

    print("\n" + "=" * 96)
    print("Structural experiments: intra-camera merge")
    print("=" * 96)
    structural_results = []
    for exp in structural_experiments:
        trial_params = dict(tuned_best_params)
        trial_params.update({
            "intra_merge": exp["intra_merge"],
            "intra_merge_thresh": exp["intra_merge_thresh"],
            "intra_merge_gap": exp["intra_merge_gap"],
        })
        result = _run_experiment("assoc_struct", exp["name"], trial_params)
        structural_results.append(result)
        all_results.append(result)

    _print_table("Structural experiment results", structural_results)

    overall_candidates = [tuned_best_result] + structural_results
    best_overall = max(overall_candidates, key=_rank_key)
    print("\n" + "=" * 96)
    print("Overall best configuration")
    print("=" * 96)
    print(
        f"run={best_overall['run_name']}\n"
        f"sim_thresh={best_overall['sim_thresh']:.2f}, appearance_weight={best_overall['appearance_w']:.2f}, "
        f"spatiotemporal_weight={round(1.0 - best_overall['appearance_w'] - HSV_WEIGHT, 4):.4f}, "
        f"fic_reg={best_overall['fic_reg']:.2f}, aqe_k={best_overall['aqe_k']}, "
        f"gallery_thresh={best_overall['gallery_thresh']:.2f}, orphan_match={best_overall['orphan_thresh']:.2f}, "
        f"aflink_gap={best_overall['aflink_max_spatial_gap']:.1f}, aflink_dir={best_overall['aflink_min_direction_cos']:.2f}, "
        f"intra_merge={best_overall['intra_merge']}, intra_thresh={best_overall['intra_merge_thresh']:.2f}, "
        f"intra_gap={best_overall['intra_merge_gap']}\n"
        f"{best_overall['score_metric']}={best_overall['score']:.3f}, IDF1={best_overall['IDF1']:.3f}, "
        f"MOTA={best_overall['MOTA']:.3f}, HOTA={best_overall['HOTA']:.3f}"
    )

    results_payload = {
        "primary_metric": PRIMARY_METRIC,
        "base_params": {
            "sim_thresh": SIM_THRESH,
            "appearance_w": APPEARANCE_WEIGHT,
            "fic_reg": BASE_FIC_REG,
            "aqe_k": AQE_K,
            "gallery_thresh": BASE_GALLERY_THRESH,
            "orphan_thresh": BASE_ORPHAN_THRESH,
            "intra_merge": INTRA_MERGE,
            "intra_merge_thresh": INTRA_MERGE_THRESH,
            "intra_merge_gap": INTRA_MERGE_GAP,
            "reranking": RERANKING,
            "reranking_k1": RERANKING_K1,
            "reranking_k2": RERANKING_K2,
            "reranking_lambda": RERANKING_LAMBDA,
            "camera_pair_norm": CAMERA_PAIR_NORM,
            "camera_pair_norm_min_pairs": 10,
            "aflink_max_spatial_gap": 200.0,
            "aflink_min_direction_cos": 0.7,
            "fusion_weight": FUSION_WEIGHT,
            "fusion_weight_tertiary": FUSION_WEIGHT_TERTIARY,
        },
        "stage_winners": stage_winners,
        "tuned_best": tuned_best_result,
        "structural_results": structural_results,
        "best_overall": best_overall,
        "results": all_results,
    }
    results_path = DATA_OUT / "association_sweep_results.json"
    with open(results_path, "w", encoding="utf-8") as handle:
        json.dump(results_payload, handle, indent=2)
    print(f"Saved sweep results to {results_path}")

    kaggle_out = Path("/kaggle/working")
    if kaggle_out.exists():
        _shutil.copy2(str(results_path), str(kaggle_out / "association_sweep_results.json"))
        print(f"Copied association_sweep_results.json to {kaggle_out}")
        best_report = DATA_OUT / best_overall["run_name"] / "stage5" / "evaluation_report.json"
        if best_report.exists():
            _shutil.copy2(str(best_report), str(kaggle_out / "evaluation_report_best.json"))
            print(f"Copied best evaluation report to {kaggle_out}")
else:
    print("Feature sweep disabled. Set FEATURE_TEST_ENABLED = True to run.")

## 5. Solver Comparison

Runs the v52 baseline parameters twice, once with `solver=cc` and once with `solver=network_flow`, so the only structural change is the Stage 4 solver.

In [ ]:
import math

BASELINE_SOLVER_RUNS = [
    {"solver": "cc", "graph_algorithm": "conflict_free_cc"},
    {"solver": "network_flow", "graph_algorithm": "conflict_free_cc"},
]
BASELINE_STAGE4 = {
    "sim_thresh": 0.50,
    "appearance": 0.70,
    "hsv": 0.0,
    "st": 0.30,
    "fic_reg": 0.50,
    "aqe_k": 3,
    "gallery_thresh": 0.48,
    "orphan_thresh": 0.38,
}

def _solver_cmp_run_name(solver_name):
    return f"{RUN_NAME}-solver-{solver_name}"

def _extract_solver_metrics(run_name):
    report_path = DATA_OUT / run_name / "stage5" / "evaluation_report.json"
    if not report_path.exists():
        return {}
    if "_load_metrics" in globals():
        return _load_metrics(report_path)
    with report_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

comparison_results = []
for variant in BASELINE_SOLVER_RUNS:
    run_name = _solver_cmp_run_name(variant["solver"])
    _ensure_upstream_links(run_name)
    cmd = [
        sys.executable, "scripts/run_pipeline.py",
        "--config", "configs/default.yaml",
        "--dataset-config", "configs/datasets/cityflowv2.yaml",
        "--stages", "4,5",
        "--override", f"project.run_name={run_name}",
        "--override", f"project.output_dir={DATA_OUT}",
        "--override", f"stage4.association.solver={variant['solver']}",
        "--override", f"stage4.association.graph.algorithm={variant['graph_algorithm']}",
        "--override", f"stage4.association.query_expansion.k={BASELINE_STAGE4['aqe_k']}",
        "--override", "stage4.association.query_expansion.alpha=5.0",
        "--override", "stage4.association.query_expansion.dba=false",
        "--override", f"stage4.association.graph.similarity_threshold={BASELINE_STAGE4['sim_thresh']}",
        "--override", "stage4.association.fic.enabled=true",
        "--override", f"stage4.association.fic.regularisation={BASELINE_STAGE4['fic_reg']}",
        "--override", f"stage4.association.weights.vehicle.appearance={BASELINE_STAGE4['appearance']}",
        "--override", f"stage4.association.weights.vehicle.hsv={BASELINE_STAGE4['hsv']}",
        "--override", f"stage4.association.weights.vehicle.spatiotemporal={BASELINE_STAGE4['st']}",
        "--override", "stage4.association.mutual_nn.top_k_per_query=20",
        "--override", "stage4.association.fac.enabled=false",
        "--override", "stage4.association.reranking.enabled=false",
        "--override", "stage4.association.intra_camera_merge.enabled=true",
        "--override", "stage4.association.intra_camera_merge.threshold=0.80",
        "--override", "stage4.association.intra_camera_merge.max_time_gap=30",
        "--override", "stage4.association.gallery_expansion.enabled=true",
        "--override", f"stage4.association.gallery_expansion.threshold={BASELINE_STAGE4['gallery_thresh']}",
        "--override", f"stage4.association.gallery_expansion.orphan_match_threshold={BASELINE_STAGE4['orphan_thresh']}",
        "--override", "stage4.association.temporal_overlap.enabled=true",
        "--override", "stage4.association.temporal_overlap.bonus=0.05",
        "--override", "stage4.association.temporal_overlap.max_mean_time=5.0",
        "--override", "stage5.mtmc_only_submission=false",
        "--override", "stage5.stationary_filter.enabled=true",
        "--override", "stage5.stationary_filter.min_displacement_px=150",
        "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
        "--override", "stage5.min_submission_confidence=0.15",
        "--override", "stage5.cross_id_nms_iou=0.40",
        "--override", "stage5.min_trajectory_confidence=0.30",
        "--override", "stage5.min_trajectory_frames=40",
        "--override", "stage5.track_edge_trim.enabled=false",
        "--override", "stage5.track_smoothing.enabled=false",
        "--override", "stage5.gt_frame_clip=true",
        "--override", "stage5.gt_zone_filter=true",
    ]
    if GT_DIR:
        cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]

    print("=" * 90)
    print(f"Running solver comparison for {variant['solver']}")
    print(" ".join(str(part) for part in cmd))
    started = time.time()
    result = subprocess.run(cmd, cwd=str(PROJECT))
    elapsed = time.time() - started
    if result.returncode != 0:
        raise RuntimeError(f"solver={variant['solver']} failed with exit code {result.returncode}")
    metrics = _extract_solver_metrics(run_name)
    comparison_results.append({
        "solver": variant["solver"],
        "time_min": elapsed / 60.0,
        **metrics,
    })

print("\n" + "=" * 90)
print("Solver comparison summary")
print("=" * 90)
for row in comparison_results:
    print(
        f"solver={row['solver']:<12} time={row['time_min']:.1f} min  "
        f"MTMC_IDF1={row.get('MTMC_IDF1', math.nan):.3f}  "
        f"IDF1={row.get('IDF1', math.nan):.3f}  "
        f"MOTA={row.get('MOTA', math.nan):.3f}  "
        f"HOTA={row.get('HOTA', math.nan):.3f}"
    )

## CID_BIAS Topology Test

Creates the topology-based CID_BIAS matrix inline, validates the runtime camera IDs from Stage 2 metadata, then compares a no-bias control against conservative, default, and aggressive topology bias settings.

In [ ]:
# =========== CID_BIAS Topology Test ===========
# Test the topology-based camera bias matrix against a no-bias control.

import json
import math
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd

CID_BIAS_EXPECTED_CAMERAS = [
    "S01_c001",
    "S01_c002",
    "S01_c003",
    "S02_c006",
    "S02_c007",
    "S02_c008",
]
CID_BIAS_PATH = PROJECT / "configs" / "datasets" / "cityflowv2_cid_bias.npy"
CID_BIAS_JSON_PATH = CID_BIAS_PATH.with_suffix(".json")
CID_BIAS_REL_PATH = CID_BIAS_PATH.relative_to(PROJECT).as_posix()
CID_BIAS_BASELINE_STAGE4 = globals().get(
    "BASELINE_STAGE4",
    {
        "sim_thresh": 0.50,
        "appearance": 0.70,
        "hsv": 0.0,
        "st": 0.30,
        "fic_reg": 0.50,
        "aqe_k": 3,
        "gallery_thresh": 0.48,
        "orphan_thresh": 0.38,
    },
)
CID_BIAS_VARIANTS = [
    {"label": "control", "enabled": False, "intra": 0.00, "cross": 0.00},
    {"label": "conservative", "enabled": True, "intra": 0.02, "cross": -0.10},
    {"label": "default", "enabled": True, "intra": 0.04, "cross": -0.15},
    {"label": "aggressive", "enabled": True, "intra": 0.06, "cross": -0.20},
]


def _cid_bias_detect_camera_order():
    index_path = RUN_DIR / "stage2" / "embedding_index.json"
    actual = []
    if index_path.exists():
        rows = json.loads(index_path.read_text(encoding="utf-8"))
        seen = set()
        for row in rows:
            camera_id = str(row.get("camera_id", "")).strip()
            if camera_id and camera_id not in seen:
                actual.append(camera_id)
                seen.add(camera_id)
        print(f"Stage2 camera IDs from {index_path.name}: {actual}")
    else:
        print(f"WARNING: {index_path} not found; falling back to benchmark camera order")

    ordered = [cam for cam in CID_BIAS_EXPECTED_CAMERAS if cam in actual]
    extras = [cam for cam in actual if cam not in CID_BIAS_EXPECTED_CAMERAS]
    if ordered or extras:
        camera_order = ordered + extras
    else:
        camera_order = list(CID_BIAS_EXPECTED_CAMERAS)

    missing = [cam for cam in CID_BIAS_EXPECTED_CAMERAS if cam not in camera_order]
    if missing:
        print(f"WARNING: missing expected benchmark cameras in Stage2 metadata: {missing}")
        print("         Unmapped cameras keep a zero CID_BIAS entry.")

    return camera_order


def _write_cid_bias_matrix(camera_order, intra_bias, cross_bias):
    n = len(camera_order)
    bias_matrix = np.zeros((n, n), dtype=np.float32)

    s01_idx = [idx for idx, cam in enumerate(camera_order) if cam.startswith("S01_")]
    s02_idx = [idx for idx, cam in enumerate(camera_order) if cam.startswith("S02_")]

    for group in (s01_idx, s02_idx):
        for i in group:
            for j in group:
                if i != j:
                    bias_matrix[i, j] = intra_bias

    for i in s01_idx:
        for j in s02_idx:
            bias_matrix[i, j] = cross_bias
            bias_matrix[j, i] = cross_bias

    CID_BIAS_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(CID_BIAS_PATH, bias_matrix)
    with CID_BIAS_JSON_PATH.open("w", encoding="utf-8") as handle:
        json.dump({"cameras": camera_order}, handle, indent=2)

    return bias_matrix


def _cid_bias_ensure_upstream_links(run_name):
    if "_ensure_upstream_links" in globals():
        return _ensure_upstream_links(run_name)

    run_dir = DATA_OUT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    for stage_sub in ("stage1", "stage2", "stage3"):
        src = RUN_DIR / stage_sub
        dst = run_dir / stage_sub
        if src.exists() and not dst.exists():
            dst.symlink_to(src)
    return run_dir


def _cid_bias_load_metrics(report_path):
    metrics = {
        "MTMC_IDF1": math.nan,
        "IDF1": math.nan,
        "MOTA": math.nan,
        "HOTA": math.nan,
    }
    if not report_path.exists():
        return metrics

    payload = json.loads(report_path.read_text(encoding="utf-8"))
    raw_metrics = payload.get("metrics", payload)
    metrics.update({
        "MTMC_IDF1": raw_metrics.get("mtmc_idf1", raw_metrics.get("MTMC_IDF1", raw_metrics.get("IDF1", math.nan))),
        "IDF1": raw_metrics.get("IDF1", raw_metrics.get("idf1", math.nan)),
        "MOTA": raw_metrics.get("MOTA", raw_metrics.get("mota", math.nan)),
        "HOTA": raw_metrics.get("HOTA", raw_metrics.get("hota", math.nan)),
    })
    return metrics


def _cid_bias_run_name(label):
    return f"{RUN_NAME}-cid-bias-{label}"


def _cid_bias_command(run_name, enabled):
    cmd = [
        sys.executable, "scripts/run_pipeline.py",
        "--config", "configs/default.yaml",
        "--dataset-config", "configs/datasets/cityflowv2.yaml",
        "--stages", "4,5",
        "--override", f"project.run_name={run_name}",
        "--override", f"project.output_dir={DATA_OUT}",
        "--override", "stage4.association.solver=cc",
        "--override", "stage4.association.graph.algorithm=conflict_free_cc",
        "--override", f"stage4.association.query_expansion.k={CID_BIAS_BASELINE_STAGE4['aqe_k']}",
        "--override", "stage4.association.query_expansion.alpha=5.0",
        "--override", "stage4.association.query_expansion.dba=false",
        "--override", f"stage4.association.graph.similarity_threshold={CID_BIAS_BASELINE_STAGE4['sim_thresh']}",
        "--override", "stage4.association.fic.enabled=true",
        "--override", f"stage4.association.fic.regularisation={CID_BIAS_BASELINE_STAGE4['fic_reg']}",
        "--override", f"stage4.association.weights.vehicle.appearance={CID_BIAS_BASELINE_STAGE4['appearance']}",
        "--override", f"stage4.association.weights.vehicle.hsv={CID_BIAS_BASELINE_STAGE4['hsv']}",
        "--override", f"stage4.association.weights.vehicle.spatiotemporal={CID_BIAS_BASELINE_STAGE4['st']}",
        "--override", "stage4.association.mutual_nn.top_k_per_query=20",
        "--override", "stage4.association.fac.enabled=false",
        "--override", "stage4.association.reranking.enabled=false",
        "--override", "stage4.association.intra_camera_merge.enabled=true",
        "--override", "stage4.association.intra_camera_merge.threshold=0.80",
        "--override", "stage4.association.intra_camera_merge.max_time_gap=30",
        "--override", "stage4.association.gallery_expansion.enabled=true",
        "--override", f"stage4.association.gallery_expansion.threshold={CID_BIAS_BASELINE_STAGE4['gallery_thresh']}",
        "--override", f"stage4.association.gallery_expansion.orphan_match_threshold={CID_BIAS_BASELINE_STAGE4['orphan_thresh']}",
        "--override", "stage4.association.temporal_overlap.enabled=true",
        "--override", "stage4.association.temporal_overlap.bonus=0.05",
        "--override", "stage4.association.temporal_overlap.max_mean_time=5.0",
        "--override", f"stage4.association.camera_bias.enabled={str(enabled).lower()}",
        "--override", "stage4.association.camera_bias.iterations=0",
        "--override", f"stage4.association.camera_bias.cid_bias_npy_path={CID_BIAS_REL_PATH}",
        "--override", "stage5.mtmc_only_submission=false",
        "--override", "stage5.stationary_filter.enabled=true",
        "--override", "stage5.stationary_filter.min_displacement_px=150",
        "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
        "--override", "stage5.min_submission_confidence=0.15",
        "--override", "stage5.cross_id_nms_iou=0.40",
        "--override", "stage5.min_trajectory_confidence=0.30",
        "--override", "stage5.min_trajectory_frames=40",
        "--override", "stage5.track_edge_trim.enabled=false",
        "--override", "stage5.track_smoothing.enabled=false",
        "--override", "stage5.gt_frame_clip=true",
        "--override", "stage5.gt_zone_filter=true",
    ]
    if GT_DIR:
        cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
    return cmd


camera_order = _cid_bias_detect_camera_order()
print(f"CID_BIAS camera order: {camera_order}")
print(f"CID_BIAS file path: {CID_BIAS_PATH}")

results = []
for variant in CID_BIAS_VARIANTS:
    run_name = _cid_bias_run_name(variant["label"])
    _cid_bias_ensure_upstream_links(run_name)

    if variant["enabled"]:
        bias_matrix = _write_cid_bias_matrix(camera_order, variant["intra"], variant["cross"])
        print("=" * 96)
        print(
            f"Testing {variant['label']} CID_BIAS: intra={variant['intra']:+.2f}, "
            f"cross={variant['cross']:+.2f}, shape={bias_matrix.shape}"
        )
    else:
        print("=" * 96)
        print("Testing control run with stage4.association.camera_bias.enabled=false")

    cmd = _cid_bias_command(run_name, enabled=variant["enabled"])
    print(" ".join(str(part) for part in cmd))
    started = time.time()
    result = subprocess.run(cmd, cwd=str(PROJECT), capture_output=True, text=True)
    elapsed_min = (time.time() - started) / 60.0
    if result.returncode != 0:
        print("stdout tail:")
        for line in result.stdout.splitlines()[-20:]:
            print(f"  {line}")
        print("stderr tail:")
        for line in result.stderr.splitlines()[-20:]:
            print(f"  {line}")
        raise RuntimeError(f"CID_BIAS variant {variant['label']} failed with exit code {result.returncode}")

    metrics = _cid_bias_load_metrics(DATA_OUT / run_name / "stage5" / "evaluation_report.json")
    row = {
        "label": variant["label"],
        "camera_bias_enabled": variant["enabled"],
        "intra_bias": variant["intra"],
        "cross_bias": variant["cross"],
        "time_min": elapsed_min,
        **metrics,
    }
    results.append(row)
    print(
        f"[{variant['label']}] MTMC_IDF1={row['MTMC_IDF1']:.3f} "
        f"IDF1={row['IDF1']:.3f} MOTA={row['MOTA']:.3f} HOTA={row['HOTA']:.3f} "
        f"({elapsed_min:.1f} min)"
    )

control_score = next((row["MTMC_IDF1"] for row in results if row["label"] == "control"), math.nan)
for row in results:
    row["delta_vs_control"] = row["MTMC_IDF1"] - control_score if not math.isnan(control_score) else math.nan

summary_df = pd.DataFrame(results)
summary_df = summary_df[
    [
        "label",
        "camera_bias_enabled",
        "intra_bias",
        "cross_bias",
        "MTMC_IDF1",
        "delta_vs_control",
        "IDF1",
        "MOTA",
        "HOTA",
        "time_min",
    ]
].sort_values(by="MTMC_IDF1", ascending=False, na_position="last")

print("\n" + "=" * 96)
print("CID_BIAS topology comparison summary")
print("=" * 96)
print(summary_df.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

payload = {
    "camera_order": camera_order,
    "cid_bias_path": str(CID_BIAS_PATH),
    "variants": results,
}
results_path = DATA_OUT / "cid_bias_topology_results.json"
results_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Saved CID_BIAS topology results to {results_path}")

kaggle_out = Path("/kaggle/working")
if kaggle_out.exists():
    shutil.copy2(str(results_path), str(kaggle_out / "cid_bias_topology_results.json"))
    shutil.copy2(str(CID_BIAS_PATH), str(kaggle_out / CID_BIAS_PATH.name))
    shutil.copy2(str(CID_BIAS_JSON_PATH), str(kaggle_out / CID_BIAS_JSON_PATH.name))
    print(f"Copied CID_BIAS artifacts to {kaggle_out}")